# Phase 5: Targeted Analysis 14: Boundless DAS & Subspace Analysis

## Overview

NB13 established where band-specific information is causally processed via
vanilla (axis-aligned) activation patching. This notebook uses **Boundless
Distributed Alignment Search (Boundless DAS)** to find the minimal linear
subspace that encodes band-distinguishing information.

Boundless DAS (Wu et al., 2024) jointly learns a rotation matrix and a
**boundary parameter** that determines the effective subspace dimension.
A sigmoid mask with temperature annealing selects which rotated dimensions
participate in the intervention. L1 regularization on the boundary encourages
the smallest sufficient subspace.

Compared to standard DAS, Boundless DAS eliminates the need for a separate
dimensionality sweep: the optimal dimension is discovered during training.

## Hypotheses

- **H1**: A low-dimensional subspace (d << d_model) achieves high IIA,
  meaning band-distinguishing information is concentrated.
- **H2**: Boundless DAS IIA >> random orthogonal IIA, confirming the subspace
  is meaningful (not axis-aligned noise).
- **H3**: DAS rotation trained on one band pair generalizes to unseen
  band pairs, confirming the subspace is universal.
- **H4**: Band classification from DAS subspace matches full activation space,
  confirming band info is concentrated in the learned subspace.

## Notebook Structure

1. Setup & Imports
2. Load NB13 Peak Layers & Data Utilities
3. Define Boundless DAS for TransformerLens
4. Train Boundless DAS (All Models, Peak Layer)
5. Boundless DAS vs Random vs Vanilla Comparison
6. Band Probing from DAS Subspace
7. Cross-Band DAS Generalization
8. Save Results
9-13. Visualizations
14. Summary

## Data Sources

- Peak layers: `outputs/analysis/activation_patching_peak_layers.json` (from NB13)
- Test data: `LSC_data/datasets/matched/{draw}/{band}/test.json`
- Reference: Wu et al. (2024), 'Boundless DAS'; old_causal implementation

In [1]:
import os
import sys
import json
import gc
from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Dict, Tuple, Optional

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import numpy as np
import pandas as pd
import torch as t
import torch.nn as nn
import torch.nn.functional as F
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

# --- Paths ---
ISC_ROOT = Path(os.environ.get("PROJECT_ROOT", ".")).resolve()
LSC_DIR = ISC_ROOT / "LSC_circuits"
AUTOCIRCUIT_PATH = os.environ.get("AUTOCIRCUIT_PATH") or str(
    ISC_ROOT / "circuit_discovery" / "auto-circuit"
)
sys.path.insert(0, AUTOCIRCUIT_PATH)
sys.path.insert(0, str(LSC_DIR))

from lsc_acdc_circuit import (
    load_model,
    model_safe_name,
    get_batch_size,
    set_all_seeds,
    cleanup_gpu,
    safe_delete_model,
)

# --- Constants ---
MODELS = ["pythia-70m", "pythia-160m", "pythia-410m", "pythia-1b", "pythia-1.4b"]
BANDS = ["low", "medium", "high", "very_high", "control"]
EVAL_SEED = 123
VARIANT = "matched"
MODEL_LAYERS = {
    "pythia-70m": 6,
    "pythia-160m": 12,
    "pythia-410m": 24,
    "pythia-1b": 16,
    "pythia-1.4b": 24,
}
MODEL_HIDDEN = {
    "pythia-70m": 512,
    "pythia-160m": 768,
    "pythia-410m": 1024,
    "pythia-1b": 2048,
    "pythia-1.4b": 2048,
}

BASE = ISC_ROOT
DATA_DIR = BASE / "LSC_data"
ANALYSIS_DIR = Path("outputs/analysis")
VIZ_DIR = Path("outputs/viz")
ROTATION_DIR = ANALYSIS_DIR / "das_rotations"
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)
VIZ_DIR.mkdir(parents=True, exist_ok=True)
ROTATION_DIR.mkdir(parents=True, exist_ok=True)

N_PAIRS = 100
POSITION = 21  # last position (with BOS)

device = "cuda:0"
print(f"Device: {device}")
print(f"CUDA available: {t.cuda.is_available()}")
if t.cuda.is_available():
    print(f"GPU: {t.cuda.get_device_name(0)}")

Device: cuda:0
CUDA available: True
GPU: NVIDIA A100 80GB PCIe


## 2. Load NB13 Peak Layers & Data Utilities

In [2]:
# Load peak layers from NB13
peak_layers_file = ANALYSIS_DIR / "activation_patching_peak_layers.json"
with open(peak_layers_file) as f:
    peak_layers = json.load(f)

print("Peak layers from NB13:")
for model_name, layers in peak_layers.items():
    print(f"  {model_name}: {layers}")


def load_test_examples(band: str, draw: str) -> List[dict]:
    path = DATA_DIR / "datasets" / VARIANT / draw / band / "test.json"
    with open(path) as f:
        data = json.load(f)
    return data["examples"]


def create_batched_pairs(
    base_band: str,
    source_band: str,
    draw: str,
    n_pairs: int = N_PAIRS,
    bos_id: int = 0,
    seed: int = EVAL_SEED,
) -> Tuple[t.Tensor, t.Tensor, t.Tensor, t.Tensor]:
    """Create batched interchange pairs.

    Returns:
        base_ids: (n_pairs, seq_len) tensor
        source_ids: (n_pairs, seq_len) tensor
        base_targets: (n_pairs,) tensor of target token ids
        source_targets: (n_pairs,) tensor of target token ids
    """
    rng = np.random.default_rng(seed)
    base_examples = load_test_examples(base_band, draw)
    source_examples = load_test_examples(source_band, draw)
    n_avail = min(len(base_examples), len(source_examples))
    indices = rng.permutation(n_avail)[:n_pairs]

    base_ids_list = []
    source_ids_list = []
    base_targets = []
    source_targets = []

    for idx in indices:
        base_ex = base_examples[idx]
        source_ex = source_examples[idx]
        base_ids_list.append([bos_id] + base_ex["token_ids"])
        source_ids_list.append([bos_id] + source_ex["token_ids"])
        base_targets.append(base_ex["target_token_id"])
        source_targets.append(source_ex["target_token_id"])

    return (
        t.tensor(base_ids_list, dtype=t.long),
        t.tensor(source_ids_list, dtype=t.long),
        t.tensor(base_targets, dtype=t.long),
        t.tensor(source_targets, dtype=t.long),
    )


print("Utilities loaded.")

Peak layers from NB13:
  pythia-70m: [5, 4, 3]
  pythia-160m: [11, 10, 9]
  pythia-410m: [23, 22, 21]
  pythia-1b: [13, 14, 15]
  pythia-1.4b: [21, 19, 20]
Utilities loaded.


## 3. Define Boundless DAS for TransformerLens

Boundless DAS (Wu et al., 2024) differs from standard DAS in two ways:
1. A **boundary parameter** $\sigma \in [0,1]$ determines the effective subspace dimension
2. A **sigmoid mask** with temperature annealing selects which rotated dims participate

The mask: $m_i = \text{sigmoid}((\sigma \cdot d_{\max} - i) / \tau)$

When $\tau$ is high, the mask is soft; as $\tau \to 0$, it becomes a hard step function.
L1 regularization on $\sigma$ encourages the boundary to shrink (smaller subspace).

In [3]:
@dataclass
class BDASResult:
    """Result of Boundless DAS training."""

    model: str
    layer: int
    final_iia: float
    initial_iia: float
    max_dim: int
    effective_dim: float  # boundary * max_dim
    boundary_value: float  # raw boundary parameter
    rotation_matrix: np.ndarray = None  # (max_dim, hidden_size)
    loss_history: List[float] = field(default_factory=list)
    iia_history: List[float] = field(default_factory=list)
    boundary_history: List[float] = field(default_factory=list)


def make_bdas_hook(source_act, position, rotation, inv_rotation, mask):
    """Boundless DAS hook for TransformerLens.

    Like standard DAS but applies a sigmoid mask in the rotated space,
    so only the first ~(boundary * max_dim) dimensions are intervened on.
    """

    def hook_fn(activation, hook):
        orig_dtype = activation.dtype
        base_at_p = activation[:, position, :].float()  # (batch, hidden)
        src_at_p = source_act[:, position, :].float()  # (batch, hidden)

        # Project to rotated space
        base_rot = rotation(base_at_p)  # (batch, max_dim)
        src_rot = rotation(src_at_p)  # (batch, max_dim)

        # Apply mask: interpolate between base and source in rotated space
        # mask ~ 1 for dims < boundary*max_dim, ~ 0 for dims above
        intervened_rot = base_rot * (1 - mask) + src_rot * mask  # (batch, max_dim)

        # Unproject
        base_contrib = inv_rotation(base_rot)
        intervened_contrib = inv_rotation(intervened_rot)
        patched_at_p = (base_at_p - base_contrib + intervened_contrib).to(orig_dtype)

        # Reconstruct full sequence
        return t.cat(
            [
                activation[:, :position, :],
                patched_at_p.unsqueeze(1),
                activation[:, position + 1 :, :],
            ],
            dim=1,
        )

    return hook_fn


def train_bdas(
    model,
    base_ids,
    source_ids,
    source_targets,
    layer: int,
    hidden_size: int,
    max_dim: int = None,
    n_steps: int = 500,
    lr_rotation: float = 1e-3,
    lr_boundary: float = 1e-2,
    boundary_lambda: float = 1.0,
    temp_start: float = 50.0,
    temp_end: float = 0.1,
    position: int = POSITION,
    batch_size: int = 50,
    boundary_warmup: int = 0,
    verbose: bool = True,
) -> BDASResult:
    """Train Boundless DAS rotation + boundary.

    The model parameters are frozen; only rotation and boundary are trained.
    All pairs are batched for efficiency.
    """
    if max_dim is None:
        max_dim = min(hidden_size // 4, 128)

    dev = next(model.parameters()).device
    hook_name = f"blocks.{layer}.hook_resid_post"
    n_pairs = base_ids.shape[0]

    # Move data to device
    base_ids_dev = base_ids.to(dev)
    source_ids_dev = source_ids.to(dev)
    source_targets_dev = source_targets.to(dev)

    # Initialize learnable rotation (float32)
    rotation = nn.Linear(hidden_size, max_dim, bias=False)
    nn.init.orthogonal_(rotation.weight)
    rotation = rotation.to(device=dev, dtype=t.float32)

    inv_rotation = nn.Linear(max_dim, hidden_size, bias=False)
    inv_rotation.weight.data = rotation.weight.data.T.clone()
    inv_rotation = inv_rotation.to(device=dev, dtype=t.float32)

    # Boundary parameter: initialized to 0.5 (half of max_dim)
    boundary = nn.Parameter(t.tensor(0.5, device=dev, dtype=t.float32))

    # Dimension indices for mask computation
    dim_indices = t.arange(max_dim, device=dev, dtype=t.float32)

    # Separate optimizers with different learning rates
    optimizer = t.optim.Adam(
        [
            {
                "params": list(rotation.parameters()) + list(inv_rotation.parameters()),
                "lr": lr_rotation,
            },
            {"params": [boundary], "lr": lr_boundary},
        ]
    )

    loss_history = []
    iia_history = []
    boundary_history = []

    # Cache ALL source activations upfront (single batched forward pass)
    with t.no_grad():
        all_source_acts = []
        for i in range(0, n_pairs, batch_size):
            batch_src = source_ids_dev[i : i + batch_size]
            _, cache = model.run_with_cache(
                batch_src, prepend_bos=False, names_filter=[hook_name]
            )
            all_source_acts.append(cache[hook_name].detach())
            del cache
        source_acts = t.cat(all_source_acts, dim=0)  # (n_pairs, seq_len, hidden)
        del all_source_acts

    # Initial IIA
    with t.no_grad():
        temp = temp_start
        mask = t.sigmoid((boundary.clamp(0.001, 1.0) * max_dim - dim_indices) / temp)
        initial_iia = _compute_bdas_iia_batched(
            model,
            base_ids_dev,
            source_acts,
            source_targets_dev,
            hook_name,
            rotation,
            inv_rotation,
            mask,
            position,
            batch_size,
        )
    iia_history.append(initial_iia)
    boundary_history.append(boundary.item())

    for step in range(n_steps):
        # Temperature annealing (linear)
        temp = temp_start + (temp_end - temp_start) * (step / max(n_steps - 1, 1))

        # Mini-batch training
        perm = t.randperm(n_pairs, device=dev)
        total_loss = 0.0
        n_batches = 0

        for i in range(0, n_pairs, batch_size):
            idx = perm[i : i + batch_size]
            batch_base = base_ids_dev[idx]
            batch_source_act = source_acts[idx]
            batch_targets = source_targets_dev[idx]

            optimizer.zero_grad()

            # Recompute mask each mini-batch (boundary is trainable, graph freed after backward)
            b_clamped = boundary.clamp(0.001, 1.0)
            mask = t.sigmoid((b_clamped * max_dim - dim_indices) / temp)

            hook_fn = make_bdas_hook(
                batch_source_act, position, rotation, inv_rotation, mask
            )
            logits = model.run_with_hooks(
                batch_base, prepend_bos=False, fwd_hooks=[(hook_name, hook_fn)]
            )

            # CE loss on source targets
            last_logits = logits[:, -1, :]  # (batch, vocab)
            ce_loss = F.cross_entropy(last_logits, batch_targets)

            # L1 regularization on boundary (with warmup)
            if step >= boundary_warmup:
                reg_loss = boundary_lambda * b_clamped
            else:
                reg_loss = 0.0

            loss = ce_loss + reg_loss
            loss.backward()
            t.nn.utils.clip_grad_norm_(
                list(rotation.parameters())
                + list(inv_rotation.parameters())
                + [boundary],
                max_norm=1.0,
            )
            optimizer.step()

            total_loss += ce_loss.item()
            n_batches += 1
            del logits, last_logits

        avg_loss = total_loss / n_batches
        loss_history.append(avg_loss)
        boundary_history.append(boundary.item())

        # Periodic evaluation
        if (step + 1) % 100 == 0:
            with t.no_grad():
                mask_eval = t.sigmoid(
                    (boundary.clamp(0.001, 1.0) * max_dim - dim_indices) / temp
                )
                current_iia = _compute_bdas_iia_batched(
                    model,
                    base_ids_dev,
                    source_acts,
                    source_targets_dev,
                    hook_name,
                    rotation,
                    inv_rotation,
                    mask_eval,
                    position,
                    batch_size,
                )
                iia_history.append(current_iia)
            eff_dim = boundary.clamp(0.001, 1.0).item() * max_dim
            if verbose:
                print(
                    f"  Step {step + 1}/{n_steps}: loss={avg_loss:.4f}, "
                    f"IIA={current_iia:.3f}, eff_dim={eff_dim:.1f}, temp={temp:.1f}"
                )

    # Final evaluation with low temperature (near-hard mask)
    with t.no_grad():
        mask_final = t.sigmoid(
            (boundary.clamp(0.001, 1.0) * max_dim - dim_indices) / temp_end
        )
        final_iia = _compute_bdas_iia_batched(
            model,
            base_ids_dev,
            source_acts,
            source_targets_dev,
            hook_name,
            rotation,
            inv_rotation,
            mask_final,
            position,
            batch_size,
        )

    eff_dim = boundary.clamp(0.001, 1.0).item() * max_dim
    rotation_matrix = rotation.weight.detach().cpu().numpy()

    del source_acts
    t.cuda.empty_cache()

    return BDASResult(
        model=str(model.cfg.model_name),
        layer=layer,
        final_iia=final_iia,
        initial_iia=initial_iia,
        max_dim=max_dim,
        effective_dim=eff_dim,
        boundary_value=boundary.item(),
        rotation_matrix=rotation_matrix,
        loss_history=loss_history,
        iia_history=iia_history,
        boundary_history=boundary_history,
    )


@t.no_grad()
def _compute_bdas_iia_batched(
    model,
    base_ids,
    source_acts,
    source_targets,
    hook_name,
    rotation,
    inv_rotation,
    mask,
    position,
    batch_size,
) -> float:
    """Compute IIA with current Boundless DAS rotation+mask (batched)."""
    matches = 0
    total = 0
    n = base_ids.shape[0]

    for i in range(0, n, batch_size):
        batch_base = base_ids[i : i + batch_size]
        batch_src_act = source_acts[i : i + batch_size]
        batch_tgt = source_targets[i : i + batch_size]

        hook_fn = make_bdas_hook(batch_src_act, position, rotation, inv_rotation, mask)
        logits = model.run_with_hooks(
            batch_base, prepend_bos=False, fwd_hooks=[(hook_name, hook_fn)]
        )
        preds = logits[:, -1, :].argmax(dim=-1)
        matches += (preds == batch_tgt).sum().item()
        total += batch_tgt.shape[0]
        del logits

    return matches / total if total > 0 else 0.0


# Also define a fixed-rotation IIA for evaluation (used in comparison/generalization)
@t.no_grad()
def compute_fixed_rotation_iia(
    model,
    base_ids,
    source_ids,
    source_targets,
    hook_name,
    rotation,
    inv_rotation,
    mask,
    position,
    batch_size=50,
) -> float:
    """Compute IIA with a fixed rotation+mask. Caches source acts internally."""
    dev = next(model.parameters()).device
    n = base_ids.shape[0]
    matches = 0
    total = 0

    for i in range(0, n, batch_size):
        batch_base = base_ids[i : i + batch_size].to(dev)
        batch_src = source_ids[i : i + batch_size].to(dev)
        batch_tgt = source_targets[i : i + batch_size].to(dev)

        _, cache = model.run_with_cache(
            batch_src, prepend_bos=False, names_filter=[hook_name]
        )
        src_act = cache[hook_name]
        del cache

        hook_fn = make_bdas_hook(src_act, position, rotation, inv_rotation, mask)
        logits = model.run_with_hooks(
            batch_base, prepend_bos=False, fwd_hooks=[(hook_name, hook_fn)]
        )
        preds = logits[:, -1, :].argmax(dim=-1)
        matches += (preds == batch_tgt).sum().item()
        total += batch_tgt.shape[0]
        del logits, src_act

    return matches / total if total > 0 else 0.0


print("Boundless DAS functions defined.")

Boundless DAS functions defined.


## 4. Train Boundless DAS (All Models, Peak Layer)

Train BDAS at the top-1 peak layer per model (from NB13).
Two band pairs: low->high and high->low.

In [ ]:
BDAS_BAND_PAIRS = [("low", "high"), ("high", "low")]
BDAS_STEPS = 500

peak_results = []
saved_rotations = {}  # (model, layer, pair) -> rotation_matrix
saved_boundaries = {}  # (model, layer, pair) -> boundary_value

for model_name in MODELS:
    hidden_size = MODEL_HIDDEN[model_name]
    top_layer = peak_layers[model_name][0]  # single peak layer
    max_dim = min(hidden_size // 4, 128)

    print(f"\n{'=' * 60}")
    print(f"{model_name}: layer={top_layer}, max_dim={max_dim}, steps={BDAS_STEPS}")
    print(f"{'=' * 60}")

    model = load_model(model_name, device)
    bos_id = model.tokenizer.bos_token_id

    for base_band, source_band in BDAS_BAND_PAIRS:
        base_ids, source_ids, base_tgt, source_tgt = create_batched_pairs(
            base_band,
            source_band,
            "draw_1",
            n_pairs=N_PAIRS,
            bos_id=bos_id,
        )
        pair_label = f"{base_band}_{source_band}"

        print(f"\n  {base_band}->{source_band}:")
        result = train_bdas(
            model,
            base_ids,
            source_ids,
            source_tgt,
            layer=top_layer,
            hidden_size=hidden_size,
            max_dim=max_dim,
            n_steps=BDAS_STEPS,
            boundary_lambda=0.05,  # low reg: let rotation learn first
            boundary_warmup=100,  # no reg for first 100 steps
        )

        peak_results.append(
            {
                "model": model_name,
                "layer": top_layer,
                "base_band": base_band,
                "source_band": source_band,
                "max_dim": max_dim,
                "effective_dim": result.effective_dim,
                "boundary": result.boundary_value,
                "initial_iia": result.initial_iia,
                "final_iia": result.final_iia,
            }
        )

        # Save rotation
        key = (model_name, top_layer, pair_label)
        saved_rotations[key] = result.rotation_matrix
        saved_boundaries[key] = result.boundary_value
        rot_path = (
            ROTATION_DIR
            / f"{model_safe_name(model_name)}_L{top_layer}_{pair_label}.npz"
        )
        np.savez(
            rot_path,
            rotation=result.rotation_matrix,
            boundary=result.boundary_value,
            max_dim=max_dim,
        )

        print(
            f"    IIA: {result.initial_iia:.3f} -> {result.final_iia:.3f}, "
            f"eff_dim={result.effective_dim:.1f}/{max_dim}"
        )

    safe_delete_model(model)
    cleanup_gpu()

df_peak = pd.DataFrame(peak_results)
print(f"\nBoundless DAS peak results: {len(df_peak)} rows")
print(
    df_peak[
        [
            "model",
            "layer",
            "base_band",
            "source_band",
            "effective_dim",
            "initial_iia",
            "final_iia",
        ]
    ].to_string(index=False)
)

## 5. Boundless DAS vs Random vs Vanilla Comparison

For each model, compare at peak layer:
- BDAS IIA (learned subspace)
- Random orthogonal IIA (same effective dim)
- Vanilla full-space IIA (from NB13)

In [5]:
# Load vanilla IIA from NB13
df_vanilla = pd.read_csv(ANALYSIS_DIR / "activation_patching_resid_sweep.csv")

comparison_results = []

for model_name in MODELS:
    hidden_size = MODEL_HIDDEN[model_name]
    top_layer = peak_layers[model_name][0]
    hook_name = f"blocks.{top_layer}.hook_resid_post"

    # Get BDAS results
    das_row = df_peak[
        (df_peak["model"] == model_name)
        & (df_peak["base_band"] == "low")
        & (df_peak["source_band"] == "high")
    ]
    das_iia = das_row["final_iia"].values[0]
    eff_dim = das_row["effective_dim"].values[0]
    eff_dim_int = max(1, int(round(eff_dim)))

    print(f"\n{model_name}: comparing at layer {top_layer}, eff_dim={eff_dim_int}")
    model = load_model(model_name, device)
    bos_id = model.tokenizer.bos_token_id

    base_ids, source_ids, _, source_tgt = create_batched_pairs(
        "low",
        "high",
        "draw_1",
        n_pairs=N_PAIRS,
        bos_id=bos_id,
    )

    # Random IIA: random orthogonal rotation of same effective dim
    dev = device
    random_iias = []
    for seed in range(5):
        rng = np.random.default_rng(seed + 42)
        random_matrix = rng.standard_normal((hidden_size, eff_dim_int))
        Q, _ = np.linalg.qr(random_matrix)
        Q = Q[:, :eff_dim_int]

        rot = nn.Linear(hidden_size, eff_dim_int, bias=False)
        rot.weight.data = t.tensor(Q.T, dtype=t.float32)
        rot = rot.to(dev)
        inv_rot = nn.Linear(eff_dim_int, hidden_size, bias=False)
        inv_rot.weight.data = t.tensor(Q, dtype=t.float32)
        inv_rot = inv_rot.to(dev)

        # Full mask (all dims active)
        full_mask = t.ones(eff_dim_int, device=dev, dtype=t.float32)

        iia = compute_fixed_rotation_iia(
            model,
            base_ids,
            source_ids,
            source_tgt,
            hook_name,
            rot,
            inv_rot,
            full_mask,
            POSITION,
        )
        random_iias.append(iia)
        del rot, inv_rot

    rand_mean = float(np.mean(random_iias))
    rand_std = float(np.std(random_iias))

    # Vanilla IIA from NB13
    vanilla_row = df_vanilla[
        (df_vanilla["model"] == model_name)
        & (df_vanilla["layer"] == top_layer)
        & (df_vanilla["base_band"] == "low")
        & (df_vanilla["source_band"] == "high")
    ]
    vanilla_iia = vanilla_row["iia"].values[0] if len(vanilla_row) > 0 else np.nan

    comparison_results.append(
        {
            "model": model_name,
            "layer": top_layer,
            "effective_dim": eff_dim,
            "das_iia": das_iia,
            "random_iia_mean": rand_mean,
            "random_iia_std": rand_std,
            "vanilla_iia": vanilla_iia,
            "das_minus_random": das_iia - rand_mean,
        }
    )
    print(
        f"  BDAS={das_iia:.3f}, Random={rand_mean:.3f}+/-{rand_std:.3f}, Vanilla={vanilla_iia:.3f}"
    )

    safe_delete_model(model)
    cleanup_gpu()

df_comparison = pd.DataFrame(comparison_results)
print(f"\nComparison results:")
print(df_comparison.to_string(index=False))


pythia-70m: comparing at layer 5, eff_dim=1


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

Loaded pretrained model pythia-70m into HookedTransformer


  BDAS=0.020, Random=0.000+/-0.000, Vanilla=0.450
Moving model to device:  cpu



pythia-160m: comparing at layer 11, eff_dim=1


...


pythia-1.4b: comparing at layer 21, eff_dim=1


Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

Loaded pretrained model pythia-1.4b into HookedTransformer


  BDAS=0.020, Random=0.000+/-0.000, Vanilla=0.970
Moving model to device:  cpu



Comparison results:
      model  layer  effective_dim  das_iia  random_iia_mean  random_iia_std  vanilla_iia  das_minus_random
 pythia-70m      5          0.128     0.02              0.0             0.0         0.45              0.02
pythia-160m     11          0.128     0.02              0.0             0.0         0.95              0.02
pythia-410m     23          0.128     0.01              0.0             0.0         0.99              0.01
  pythia-1b     13          0.128     0.07              0.0             0.0         1.00              0.07
pythia-1.4b     21          0.128     0.02              0.0             0.0         0.97              0.02


## 6. Band Probing from DAS Subspace

Extract activations, project via BDAS rotation (truncated to effective dim),
classify band. Compare DAS subspace vs random subspace vs full activations.

In [6]:
probing_results = []

for model_name in MODELS:
    hidden_size = MODEL_HIDDEN[model_name]
    top_layer = peak_layers[model_name][0]
    hook_name = f"blocks.{top_layer}.hook_resid_post"

    # Get effective dim
    row = df_peak[
        (df_peak["model"] == model_name)
        & (df_peak["base_band"] == "low")
        & (df_peak["source_band"] == "high")
    ]
    eff_dim_int = max(1, int(round(row["effective_dim"].values[0])))

    print(f"\n{model_name}: probing at layer {top_layer}, eff_dim={eff_dim_int}")
    model = load_model(model_name, device)
    bos_id = model.tokenizer.bos_token_id

    # Extract activations for all bands (batched)
    all_acts = []
    all_labels = []

    for band_idx, band in enumerate(BANDS):
        examples = load_test_examples(band, "draw_1")
        batch_ids = []
        for ex in examples[:100]:
            batch_ids.append([bos_id] + ex["token_ids"])
        batch_tensor = t.tensor(batch_ids, dtype=t.long, device=device)

        # Forward in chunks
        for i in range(0, len(batch_ids), 50):
            chunk = batch_tensor[i : i + 50]
            _, cache = model.run_with_cache(
                chunk, prepend_bos=False, names_filter=[hook_name]
            )
            acts = cache[hook_name][:, -1, :].cpu().numpy()  # (chunk, hidden)
            all_acts.append(acts)
            all_labels.extend([band_idx] * acts.shape[0])
            del cache

    X_full = np.concatenate(all_acts, axis=0)  # (500, hidden_size)
    y = np.array(all_labels)
    print(f"  Extracted: {X_full.shape}")

    # Load BDAS rotation
    rot_key = (model_name, top_layer, "low_high")
    R = saved_rotations[rot_key]  # (max_dim, hidden_size)

    # Project onto DAS subspace (use first eff_dim_int rotated dims)
    X_das = X_full @ R[:eff_dim_int, :].T  # (500, eff_dim_int)

    # Random subspace (mean over 5 seeds)
    random_accs = []
    for seed in range(5):
        rng = np.random.default_rng(seed + 42)
        Q, _ = np.linalg.qr(rng.standard_normal((hidden_size, eff_dim_int)))
        X_rand = X_full @ Q[:, :eff_dim_int]
        acc = cross_val_score(
            LogisticRegression(max_iter=1000, random_state=42),
            X_rand,
            y,
            cv=5,
            scoring="accuracy",
        ).mean()
        random_accs.append(acc)

    # DAS subspace probing
    das_acc = cross_val_score(
        LogisticRegression(max_iter=1000, random_state=42),
        X_das,
        y,
        cv=5,
        scoring="accuracy",
    ).mean()

    # Full activation probing
    full_acc = cross_val_score(
        LogisticRegression(max_iter=1000, random_state=42),
        X_full,
        y,
        cv=5,
        scoring="accuracy",
    ).mean()

    probing_results.append(
        {
            "model": model_name,
            "layer": top_layer,
            "das_acc": das_acc,
            "random_acc_mean": np.mean(random_accs),
            "random_acc_std": np.std(random_accs),
            "full_acc": full_acc,
            "effective_dim": eff_dim_int,
            "full_dim": hidden_size,
        }
    )
    print(
        f"  DAS={das_acc:.3f}, Random={np.mean(random_accs):.3f}+/-{np.std(random_accs):.3f}, Full={full_acc:.3f}"
    )

    safe_delete_model(model)
    cleanup_gpu()

df_probing = pd.DataFrame(probing_results)
print(f"\nProbing results:")
print(df_probing.to_string(index=False))


pythia-70m: probing at layer 5, eff_dim=1


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

Loaded pretrained model pythia-70m into HookedTransformer


  Extracted: (500, 512)


  DAS=0.282, Random=0.212+/-0.024, Full=0.534
Moving model to device:  cpu


...

Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

Loaded pretrained model pythia-1.4b into HookedTransformer


  Extracted: (500, 2048)


  DAS=0.406, Random=0.251+/-0.041, Full=0.694
Moving model to device:  cpu



Probing results:
      model  layer  das_acc  random_acc_mean  random_acc_std  full_acc  effective_dim  full_dim
 pythia-70m      5    0.282           0.2116        0.024344     0.534              1       512
pythia-160m     11    0.228           0.2268        0.015105     0.576              1       768
pythia-410m     23    0.272           0.2284        0.015869     0.664              1      1024
  pythia-1b     13    0.406           0.2380        0.025644     0.670              1      2048
pythia-1.4b     21    0.406           0.2508        0.041388     0.694              1      2048


## 7. Cross-Band DAS Generalization

Use the rotation trained on low->high to intervene on unseen band pairs.
If the learned subspace generalizes, it is universal: not pair-specific.

In [7]:
TEST_PAIRS = [
    ("low", "high"),  # training pair (sanity check)
    ("high", "low"),  # reverse direction
    ("medium", "very_high"),  # unseen
    ("very_high", "medium"),  # unseen reverse
    ("low", "control"),  # vs control
    ("control", "high"),  # control->high
]

generalization_results = []

for model_name in MODELS:
    hidden_size = MODEL_HIDDEN[model_name]
    top_layer = peak_layers[model_name][0]
    hook_name = f"blocks.{top_layer}.hook_resid_post"

    # Get boundary and rotation from low->high training
    rot_key = (model_name, top_layer, "low_high")
    R = saved_rotations[rot_key]
    b_val = saved_boundaries[rot_key]
    max_dim = R.shape[0]

    print(f"\n{model_name}: generalization test at layer {top_layer}")
    model = load_model(model_name, device)
    bos_id = model.tokenizer.bos_token_id
    dev = device

    # Reconstruct rotation modules + mask
    rotation = nn.Linear(hidden_size, max_dim, bias=False)
    rotation.weight.data = t.tensor(R, dtype=t.float32)
    rotation = rotation.to(dev)
    inv_rotation = nn.Linear(max_dim, hidden_size, bias=False)
    inv_rotation.weight.data = t.tensor(R.T, dtype=t.float32)
    inv_rotation = inv_rotation.to(dev)

    dim_indices = t.arange(max_dim, device=dev, dtype=t.float32)
    mask = t.sigmoid(
        (min(max(b_val, 0.001), 1.0) * max_dim - dim_indices) / 0.1
    )  # hard mask

    for base_band, source_band in TEST_PAIRS:
        base_ids, source_ids, _, source_tgt = create_batched_pairs(
            base_band,
            source_band,
            "draw_1",
            n_pairs=N_PAIRS,
            bos_id=bos_id,
        )
        iia = compute_fixed_rotation_iia(
            model,
            base_ids,
            source_ids,
            source_tgt,
            hook_name,
            rotation,
            inv_rotation,
            mask,
            POSITION,
        )

        is_training_pair = base_band == "low" and source_band == "high"
        generalization_results.append(
            {
                "model": model_name,
                "layer": top_layer,
                "base_band": base_band,
                "source_band": source_band,
                "das_iia": iia,
                "is_training_pair": is_training_pair,
            }
        )
        marker = " (train)" if is_training_pair else ""
        print(f"  {base_band}->{source_band}: IIA={iia:.3f}{marker}")

    del rotation, inv_rotation
    safe_delete_model(model)
    cleanup_gpu()

df_gen = pd.DataFrame(generalization_results)
print(f"\nGeneralization: {len(df_gen)} rows")


pythia-70m: generalization test at layer 5


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

Loaded pretrained model pythia-70m into HookedTransformer
  low->high: IIA=0.000 (train)


  high->low: IIA=0.000
  medium->very_high: IIA=0.010


  very_high->medium: IIA=0.000
  low->control: IIA=0.000


...

  medium->very_high: IIA=0.000


  very_high->medium: IIA=0.000


  low->control: IIA=0.000


  control->high: IIA=0.000
Moving model to device:  cpu



Generalization: 30 rows


## 8. Save Results

In [8]:
df_peak.to_csv(ANALYSIS_DIR / "das_peak_layers.csv", index=False)
df_comparison.to_csv(ANALYSIS_DIR / "das_vs_random.csv", index=False)
df_probing.to_csv(ANALYSIS_DIR / "das_probing.csv", index=False)
df_gen.to_csv(ANALYSIS_DIR / "das_cross_band_generalization.csv", index=False)

print("Saved:")
for fname in [
    "das_peak_layers.csv",
    "das_vs_random.csv",
    "das_probing.csv",
    "das_cross_band_generalization.csv",
]:
    print(f"  {fname}")
print(f"  das_rotations/ ({len(list(ROTATION_DIR.glob('*.npz')))} files)")

Saved:
  das_peak_layers.csv
  das_vs_random.csv
  das_probing.csv
  das_cross_band_generalization.csv
  das_rotations/ (10 files)


## 9. Viz 1: Effective Dimension & IIA per Model

In [9]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
model_labels = [m.replace("pythia-", "") for m in MODELS]
x = np.arange(len(MODELS))
width = 0.35

# Left: effective dimension
for i, (bb, sb) in enumerate(BDAS_BAND_PAIRS):
    df_p = df_peak[(df_peak["base_band"] == bb) & (df_peak["source_band"] == sb)]
    df_p = df_p.set_index("model").loc[MODELS]
    color = "#E24A33" if i == 0 else "#348ABD"
    ax1.bar(
        x + i * width - width / 2,
        df_p["effective_dim"],
        width,
        label=f"{bb}->{sb}",
        color=color,
        alpha=0.85,
    )

ax1.set_xlabel("Model")
ax1.set_ylabel("Effective Subspace Dimension")
ax1.set_title("Boundless DAS: Learned Subspace Dimension")
ax1.set_xticks(x)
ax1.set_xticklabels(model_labels)
ax1.legend()

# Right: IIA (initial vs final)
for i, (bb, sb) in enumerate(BDAS_BAND_PAIRS):
    df_p = df_peak[(df_peak["base_band"] == bb) & (df_peak["source_band"] == sb)]
    df_p = df_p.set_index("model").loc[MODELS]
    color = "#E24A33" if i == 0 else "#348ABD"
    ax2.bar(
        x + i * width - width / 2,
        df_p["final_iia"],
        width,
        label=f"{bb}->{sb} (trained)",
        color=color,
        alpha=0.85,
    )
    ax2.bar(
        x + i * width - width / 2,
        df_p["initial_iia"],
        width,
        label=f"{bb}->{sb} (initial)" if i == 0 else None,
        color="gray",
        alpha=0.3,
    )

ax2.set_xlabel("Model")
ax2.set_ylabel("IIA")
ax2.set_title("Boundless DAS: IIA Before and After Training")
ax2.set_xticks(x)
ax2.set_xticklabels(model_labels)
ax2.legend(fontsize=8)
ax2.set_ylim(0, 1.05)

plt.tight_layout()
plt.savefig(VIZ_DIR / "T14_01_bdas_dim_and_iia.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved T14_01_bdas_dim_and_iia.png")

Saved T14_01_bdas_dim_and_iia.png


## 10. Viz 2: BDAS vs Baselines

In [10]:
fig, ax = plt.subplots(1, 1, figsize=(10, 5))
model_labels = [m.replace("pythia-", "") for m in MODELS]
x = np.arange(len(MODELS))
width = 0.25

ax.bar(
    x - width,
    df_comparison["vanilla_iia"],
    width,
    label="Vanilla (full-space)",
    color="#4C72B0",
    alpha=0.85,
)
ax.bar(
    x,
    df_comparison["random_iia_mean"],
    width,
    label=f"Random (matched dim)",
    color="#AAAAAA",
    alpha=0.85,
    yerr=df_comparison["random_iia_std"],
    capsize=3,
)
ax.bar(
    x + width,
    df_comparison["das_iia"],
    width,
    label="Boundless DAS",
    color="#E24A33",
    alpha=0.85,
)

ax.set_xlabel("Model")
ax.set_ylabel("IIA")
ax.set_title(
    "Interchange Intervention Accuracy: Boundless DAS vs Baselines (low->high, peak layer)"
)
ax.set_xticks(x)
ax.set_xticklabels(model_labels)
ax.legend()
ax.set_ylim(0, 1.05)

plt.tight_layout()
plt.savefig(VIZ_DIR / "T14_02_bdas_vs_baselines.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved T14_02_bdas_vs_baselines.png")

Saved T14_02_bdas_vs_baselines.png


## 11. Viz 3: Band Probing Comparison

In [11]:
fig, ax = plt.subplots(1, 1, figsize=(10, 5))
model_labels = [m.replace("pythia-", "") for m in MODELS]
x = np.arange(len(MODELS))
width = 0.25

ax.bar(
    x - width,
    df_probing["full_acc"],
    width,
    label=f"Full (d_model)",
    color="#4C72B0",
    alpha=0.85,
)
ax.bar(
    x,
    df_probing["random_acc_mean"],
    width,
    label=f"Random (matched dim)",
    color="#AAAAAA",
    alpha=0.85,
    yerr=df_probing["random_acc_std"],
    capsize=3,
)
ax.bar(
    x + width,
    df_probing["das_acc"],
    width,
    label="Boundless DAS",
    color="#E24A33",
    alpha=0.85,
)

ax.axhline(y=0.2, color="gray", linestyle="--", alpha=0.5, label="Chance (1/5)")

ax.set_xlabel("Model")
ax.set_ylabel("5-Fold CV Accuracy")
ax.set_title("Band Classification: DAS Subspace vs Full Activations")
ax.set_xticks(x)
ax.set_xticklabels(model_labels)
ax.legend()
ax.set_ylim(0, 1.05)

plt.tight_layout()
plt.savefig(VIZ_DIR / "T14_03_probing_comparison.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved T14_03_probing_comparison.png")

Saved T14_03_probing_comparison.png


## 12. Viz 4: Cross-Band DAS Generalization

In [12]:
fig, axes = plt.subplots(1, 5, figsize=(22, 4), sharey=True)

pair_labels = [f"{bb}->{sb}" for bb, sb in TEST_PAIRS]

for col, model_name in enumerate(MODELS):
    ax = axes[col]
    df_m = df_gen[df_gen["model"] == model_name]

    iias = []
    colors = []
    for bb, sb in TEST_PAIRS:
        row = df_m[(df_m["base_band"] == bb) & (df_m["source_band"] == sb)]
        iias.append(row["das_iia"].values[0] if len(row) > 0 else 0)
        colors.append("#E24A33" if (bb == "low" and sb == "high") else "#4C72B0")

    ax.bar(range(len(iias)), iias, color=colors, alpha=0.85)
    ax.set_xticks(range(len(pair_labels)))
    ax.set_xticklabels(pair_labels, rotation=45, ha="right", fontsize=7)
    ax.set_title(model_name.replace("pythia-", ""), fontsize=10)
    if col == 0:
        ax.set_ylabel("BDAS IIA")

plt.suptitle(
    "BDAS Generalization: Trained on low->high (red), tested on others (blue)",
    fontsize=12,
)
plt.tight_layout()
plt.savefig(VIZ_DIR / "T14_04_bdas_generalization.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved T14_04_bdas_generalization.png")

Saved T14_04_bdas_generalization.png


## 13. Viz 5: Boundary Convergence

In [13]:
# Reload from saved npz to get boundary values for the plot
fig, ax = plt.subplots(1, 1, figsize=(10, 5))

model_labels = [m.replace("pythia-", "") for m in MODELS]
x = np.arange(len(MODELS))
width = 0.35

for i, (bb, sb) in enumerate(BDAS_BAND_PAIRS):
    dims = []
    for model_name in MODELS:
        row = df_peak[
            (df_peak["model"] == model_name)
            & (df_peak["base_band"] == bb)
            & (df_peak["source_band"] == sb)
        ]
        dims.append(row["effective_dim"].values[0])

    hidden_sizes = [MODEL_HIDDEN[m] for m in MODELS]
    fractions = [d / h * 100 for d, h in zip(dims, hidden_sizes)]

    color = "#E24A33" if i == 0 else "#348ABD"
    ax.bar(
        x + i * width - width / 2,
        fractions,
        width,
        label=f"{bb}->{sb}",
        color=color,
        alpha=0.85,
    )

ax.set_xlabel("Model")
ax.set_ylabel("Effective Dim / d_model (%)")
ax.set_title("BDAS: Fraction of Activation Space Used")
ax.set_xticks(x)
ax.set_xticklabels(model_labels)
ax.legend()

plt.tight_layout()
plt.savefig(VIZ_DIR / "T14_05_bdas_fraction.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved T14_05_bdas_fraction.png")

Saved T14_05_bdas_fraction.png


## 14. Summary

In [14]:
print("=" * 70)
print("BOUNDLESS DAS SUBSPACE ANALYSIS SUMMARY")
print("=" * 70)

print(f"\n--- Learned Subspace Dimensions ---")
print(
    f"{'Model':<12} {'Layer':>5} {'Eff Dim (l->h)':>14} {'Eff Dim (h->l)':>14} {'d_model':>8}"
)
for model_name in MODELS:
    lh = df_peak[(df_peak["model"] == model_name) & (df_peak["base_band"] == "low")]
    hl = df_peak[(df_peak["model"] == model_name) & (df_peak["base_band"] == "high")]
    print(
        f"{model_name:<12} {lh['layer'].values[0]:>5} "
        f"{lh['effective_dim'].values[0]:>14.1f} "
        f"{hl['effective_dim'].values[0]:>14.1f} "
        f"{MODEL_HIDDEN[model_name]:>8}"
    )

print(f"\n--- BDAS vs Baselines (low->high, peak layer) ---")
print(f"{'Model':<12} {'BDAS':>8} {'Random':>8} {'Vanilla':>8} {'BDAS-Rand':>10}")
for _, row in df_comparison.iterrows():
    print(
        f"{row['model']:<12} {row['das_iia']:>8.3f} {row['random_iia_mean']:>8.3f} "
        f"{row['vanilla_iia']:>8.3f} {row['das_minus_random']:>+10.3f}"
    )

print(f"\n--- Band Probing (5-fold CV accuracy) ---")
print(f"{'Model':<12} {'DAS':>8} {'Random':>8} {'Full':>8}")
for _, row in df_probing.iterrows():
    print(
        f"{row['model']:<12} {row['das_acc']:>8.3f} {row['random_acc_mean']:>8.3f} {row['full_acc']:>8.3f}"
    )

print(f"\n--- Generalization (trained on low->high) ---")
for model_name in MODELS:
    df_m = df_gen[df_gen["model"] == model_name]
    train_iia = df_m[df_m["is_training_pair"]]["das_iia"].values[0]
    test_iias = df_m[~df_m["is_training_pair"]]["das_iia"]
    retention = test_iias.mean() / train_iia if train_iia > 0 else 0
    print(
        f"{model_name}: train={train_iia:.3f}, test_mean={test_iias.mean():.3f}, "
        f"retention={retention:.1%}"
    )

print("\n--- Hypothesis Verdicts ---")
mean_das = df_comparison["das_iia"].mean()
mean_rand = df_comparison["random_iia_mean"].mean()
mean_gap = df_comparison["das_minus_random"].mean()
mean_eff_dim = df_peak[df_peak["base_band"] == "low"]["effective_dim"].mean()
mean_hidden = np.mean([MODEL_HIDDEN[m] for m in MODELS])
print(
    f"H1 (low-dim subspace): mean eff_dim = {mean_eff_dim:.1f} "
    f"({mean_eff_dim / mean_hidden:.1%} of d_model), mean IIA = {mean_das:.3f}"
)
print(f"H2 (BDAS >> random): mean gap = {mean_gap:+.3f}")

gen_retention = []
for model_name in MODELS:
    df_m = df_gen[df_gen["model"] == model_name]
    train = df_m[df_m["is_training_pair"]]["das_iia"].values[0]
    test = df_m[~df_m["is_training_pair"]]["das_iia"].mean()
    if train > 0:
        gen_retention.append(test / train)
mean_retention = np.mean(gen_retention)
print(f"H3 (generalization): mean retention = {mean_retention:.1%}")

mean_das_probe = df_probing["das_acc"].mean()
mean_full_probe = df_probing["full_acc"].mean()
print(
    f"H4 (probing): DAS={mean_das_probe:.3f} vs Full={mean_full_probe:.3f} "
    f"({mean_das_probe / mean_full_probe:.1%} retention)"
)

print("\nDone.")

BOUNDLESS DAS SUBSPACE ANALYSIS SUMMARY

--- Learned Subspace Dimensions ---
Model        Layer  Eff Dim (l->h)  Eff Dim (h->l)  d_model
pythia-70m       5            0.1            0.1      512
pythia-160m     11            0.1            0.1      768


pythia-410m     23            0.1            0.1     1024
pythia-1b       13            0.1            0.1     2048
pythia-1.4b     21            0.1            0.1     2048

--- BDAS vs Baselines (low->high, peak layer) ---
Model            BDAS   Random  Vanilla  BDAS-Rand
pythia-70m      0.020    0.000    0.450     +0.020
pythia-160m     0.020    0.000    0.950     +0.020
pythia-410m     0.010    0.000    0.990     +0.010
pythia-1b       0.070    0.000    1.000     +0.070
pythia-1.4b     0.020    0.000    0.970     +0.020

--- Band Probing (5-fold CV accuracy) ---
Model             DAS   Random     Full
pythia-70m      0.282    0.212    0.534
pythia-160m     0.228    0.227    0.576
pythia-410m     0.272    0.228    0.664
pythia-1b       0.406    0.238    0.670
pythia-1.4b     0.406    0.251    0.694

--- Generalization (trained on low->high) ---
pythia-70m: train=0.000, test_mean=0.002, retention=0.0%
pythia-160m: train=0.000, test_mean=0.000, retention=0.0%
pythia-410m: train=0.000

<TMPDIR>/env/lib/python3.12/site-packages/numpy/core/fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
<TMPDIR>/env/lib/python3.12/site-packages/numpy/core/_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
